# YOLOv8 Detection Modules

**Purpose:** Shared modules for all YOLOv8 detection experiments.

**Usage:** Run this notebook first, then use `%run ./YOLOv8_modules.ipynb` in experiment notebooks.

## Cell 1: Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from ultralytics.nn.modules import C2f
import yaml
import cv2
import copy
from typing import Dict, List, Optional
from tqdm import tqdm

%matplotlib inline

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Cell 2: YOLOv8Detector Model

In [ ]:
class YOLOv8Detector(nn.Module):
    """
    YOLOv8 detector with configurable backbone and customization support.
    """
    
    def __init__(self, backbone: str = 'm', input_size: int = 640, 
                 confidence_threshold: float = 0.5, nms_iou_threshold: float = 0.45,
                 pretrained: bool = True, customize_type: Optional[str] = None):
        """
        Initialize YOLOv8 detector.
        
        Args:
            backbone: Model size ('n', 's', 'm', 'l', 'x')
            input_size: Input image size
            confidence_threshold: Detection confidence threshold
            nms_iou_threshold: NMS IoU threshold
            pretrained: Use pretrained weights
            customize_type: Type of customization ('deeper', 'shallower', or None)
        """
        super(YOLOv8Detector, self).__init__()
        
        self.backbone = backbone
        self.input_size = input_size
        self.confidence_threshold = confidence_threshold
        self.nms_iou_threshold = nms_iou_threshold
        self.customize_type = customize_type
        
        # Load YOLOv8 model
        model_name = f'yolov8{backbone}.pt' if pretrained else f'yolov8{backbone}.yaml'
        self.model = YOLO(model_name)
        
        # Apply customization after loading
        if customize_type == 'deeper':
            self._add_conv_layers()
        elif customize_type == 'shallower':
            self._reduce_conv_layers()
        
        # Set thresholds
        self.model.model.conf = confidence_threshold
        self.model.model.iou = nms_iou_threshold
    
    def _add_conv_layers(self):
        """
        V2: Add extra convolutional layers to the backbone after layer2.
        Purpose: Deepen shallow-layer feature extraction.
        """
        try:
            device = next(self.model.model.parameters()).device
            
            # Create new Conv layers (after backbone index 2)
            new_conv_1 = nn.Sequential(
                nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.SiLU(inplace=True)
            ).to(device)
            
            new_conv_2 = nn.Sequential(
                nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.SiLU(inplace=True)
            ).to(device)
            
            # Store original layers
            original_layers = list(self.model.model.model.children())
            
            # Create new model with inserted layers
            new_layers = []
            
            # Copy layers 0-2 (keep as is)
            for i in range(3):
                new_layers.append(original_layers[i])
            
            # Add new conv layers
            new_layers.append(new_conv_1)
            new_layers.append(new_conv_2)
            
            # Copy remaining layers
            for i in range(3, len(original_layers)):
                new_layers.append(original_layers[i])
            
            # Replace the model's sequential with new layers
            self.model.model.model = nn.Sequential(*new_layers)
            
            print("✅ Added 2 convolutional layers to backbone (deeper model)")
            
        except Exception as e:
            print(f"⚠️ Warning: Could not add conv layers: {e}")
            print("Continuing with standard model...")
    
    def _reduce_conv_layers(self):
        """
        V3: Reduce convolutional layers in the backbone by modifying C2f module.
        Purpose: Create a lighter model with faster inference.
        """
        try:
            backbone = self.model.model.model
            
            # Target C2f module at backbone index 6 (P4/16 level)
            target_idx = 6
            target_layer = backbone[target_idx]
            
            if not isinstance(target_layer, C2f):
                raise ValueError(f"Expected C2f at backbone index {target_idx}, found {type(target_layer)}")
            
            c1 = target_layer.cv1.conv.in_channels
            c2 = target_layer.cv2.conv.out_channels
            original_n = len(target_layer.m)
            
            # Compute expansion ratio
            e = target_layer.cv1.conv.out_channels / (2 * c2)
            shortcut = getattr(target_layer.m[0], 'shortcut', False)
            g = target_layer.cv1.conv.groups
            
            # New reduced repeat count: half the original
            new_n = max(1, original_n // 2)
            reduced_layer = C2f(c1, c2, n=new_n, shortcut=shortcut, g=g, e=e)
            
            # Replace the layer in-place
            backbone[target_idx] = reduced_layer
            self.model.model.model = backbone
            
            print(f"✅ Replaced C2f at backbone index {target_idx}: original n={original_n}, new n={new_n}")
            print("✅ Shallower backbone configured successfully")
            
        except Exception as e:
            print(f"⚠️ Warning: Could not reduce conv layers: {e}")
            print("Continuing with standard model...")
    
    def forward(self, x, **kwargs):
        """Forward pass."""
        return self.model(x, verbose=False, **kwargs)
    
    def train_model(self, data: str, epochs: int = 100, imgsz: int = None, **kwargs):
        """
        Train the model.
        
        Args:
            data: Dataset config path
            epochs: Number of epochs
            imgsz: Image size
            **kwargs: Additional training arguments
        """
        if imgsz is None:
            imgsz = self.input_size
        
        results = self.model.train(
            data=data,
            epochs=epochs,
            imgsz=imgsz,
            verbose=True,
            **kwargs
        )
        return results
    
    def save(self, save_path: str):
        """Save model."""
        self.model.save(save_path)

print("✓ YOLOv8Detector defined")

## Cell 3: Model Configurations

In [ ]:
# Model configurations
YOLOV8_BASELINE_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': None  # Baseline - no customization
}

YOLOV8_V2_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': 'deeper'  # Adds conv layers to backbone
}

YOLOV8_V3_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': 'shallower'  # Reduces conv layers in backbone
}

print("✓ Model configurations defined")

## Cell 4: YOLOv8Trainer

In [ ]:
class YOLOv8Trainer:
    """
    Simple trainer for YOLOv8 models using Ultralytics framework.
    """
    
    def __init__(self, learning_rate: float = 0.001, batch_size: int = 16,
                 epochs: int = 100, optimizer: str = 'adam', 
                 weight_decay: float = 1e-4, use_amp: bool = True,
                 patience: int = 15, cos_lr: bool = False, close_mosaic: int = 0):
        """
        Initialize YOLOv8 trainer.
        
        Args:
            learning_rate: Initial learning rate
            batch_size: Batch size
            epochs: Number of epochs
            optimizer: Optimizer type ('adam', 'sgd', 'adamw')
            weight_decay: L2 regularization
            use_amp: Use mixed precision training
            patience: Early stopping patience
            cos_lr: Use cosine learning rate scheduler
            close_mosaic: Stop mosaic augmentation N epochs before end
        """
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.optimizer = optimizer
        self.weight_decay = weight_decay
        self.use_amp = use_amp
        self.patience = patience
        self.cos_lr = cos_lr
        self.close_mosaic = close_mosaic
    
    def train(self, model, train_data: str, val_data: str, 
             output_dir: str, **kwargs):
        """
        Train YOLOv8 model.
        
        Args:
            model: YOLOv8Detector instance
            train_data: Training dataset config path
            val_data: Validation dataset config path
            output_dir: Directory to save outputs
            **kwargs: Additional training arguments
        
        Returns:
            Training results object from Ultralytics
        """
        print(f'\nTraining Configuration:')
        print(f'  Learning Rate: {self.learning_rate}')
        print(f'  Batch Size: {self.batch_size}')
        print(f'  Epochs: {self.epochs}')
        print(f'  Optimizer: {self.optimizer}')
        print(f'  Weight Decay: {self.weight_decay}')
        print(f'  Mixed Precision: {self.use_amp}')
        print(f'  Early Stopping Patience: {self.patience}')
        print(f'  Cosine LR: {self.cos_lr}')
        print(f'  Close Mosaic: {self.close_mosaic} epochs before end\n')
        
        # Train using Ultralytics framework
        results = model.train_model(
            data=train_data,
            epochs=self.epochs,
            batch=self.batch_size,
            lr0=self.learning_rate,
            optimizer=self.optimizer,
            weight_decay=self.weight_decay,
            amp=self.use_amp,
            patience=self.patience,
            cos_lr=self.cos_lr,
            close_mosaic=self.close_mosaic,
            project=output_dir,
            name='train',
            exist_ok=True,
            **kwargs
        )
        
        return results

print("✓ YOLOv8Trainer defined")

## Cell 5: Training Configurations

In [ ]:
# Training configurations for each experiment
YOLOV8_V1_CONFIG = {
    # Baseline Configuration
    'learning_rate': 0.001,
    'batch_size': 16,
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 1e-4,
    'use_amp': True,
    'patience': 50,
    'cos_lr': False,
    'close_mosaic': 0
}

YOLOV8_V2_CONFIG = {
    # Deeper Backbone Configuration
    'learning_rate': 0.0005,  # Lower LR for deeper model stability
    'batch_size': 12,         # Smaller batch due to larger model
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 5e-4,     # Higher weight decay to prevent overfitting
    'use_amp': True,
    'patience': 50,
    'cos_lr': True,           # Cosine LR schedule for better convergence
    'close_mosaic': 10        # Close mosaic in last 10 epochs
}

YOLOV8_V3_CONFIG = {
    # Shallower Backbone Configuration
    'learning_rate': 0.001,
    'batch_size': 20,         # Larger batch possible due to smaller model
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 1e-4,
    'use_amp': True,
    'patience': 50,
    'cos_lr': False,
    'close_mosaic': 0
}

print("✓ Training configurations defined")

## Cell 6: DetectionEvaluator

In [ ]:
class DetectionEvaluator:
    """
    Evaluator for YOLOv8 detection models.
    """
    
    def __init__(self):
        self.metrics = {}
    
    def evaluate_yolov8(self, model, test_dataset, output_dir=None):
        """
        Evaluate YOLOv8 model on test dataset.
        
        Args:
            model: YOLOv8Detector instance
            test_dataset: Test dataset config path
            output_dir: IGNORED (no file saving)
        
        Returns:
            dict: Evaluation metrics including mAP, precision, recall, F1
        """
        print("=" * 80)
        print("MODEL EVALUATION")
        print("=" * 80)
        
        # Use Ultralytics built-in validation
        results = model.model.val(
            data=test_dataset,
            split='test',
            save_json=False,
            save_hybrid=False,
            verbose=True
        )
        
        # Extract metrics
        try:
            map50 = results.box.map50
            map50_95 = results.box.map
            precision = results.box.mp
            recall = results.box.mr
            f1 = results.box.f1
        except AttributeError:
            # Fallback for different Ultralytics versions
            map50 = results.metrics.get('map50', 0)
            map50_95 = results.metrics.get('map', 0)
            precision = results.metrics.get('precision', 0)
            recall = results.metrics.get('recall', 0)
            f1 = results.metrics.get('f1', 0)
        
        metrics = {
            'map50': float(map50),
            'map50_95': float(map50_95),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1)
        }
        
        self.metrics = metrics
        
        print(f'\n=== Test Set Metrics ===')
        print(f"mAP@0.5: {metrics['map50']:.4f}")
        print(f"mAP@0.5:0.95: {metrics['map50_95']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1-Score: {metrics['f1']:.4f}")
        
        return metrics
    
    def plot_detection_results(self, model, test_dataset, num_samples=5):
        """
        Plot detection results with bounding boxes inline.
        
        Args:
            model: YOLOv8Detector instance
            test_dataset: Dataset config path
            num_samples: Number of sample images to display
        
        Returns:
            matplotlib figure
        """
        # Load dataset config to get test image paths
        with open(test_dataset, 'r') as f:
            config = yaml.safe_load(f)
        
        test_path = config.get('test', config.get('val', ''))
        if not test_path:
            print("Warning: Test path not found in dataset config")
            return None
        
        # Get image files
        test_dir = Path(test_path)
        if test_dir.is_file():
            # If it's a txt file with image paths
            with open(test_dir, 'r') as f:
                image_paths = [line.strip() for line in f.readlines()[:num_samples]]
        else:
            # If it's a directory
            image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
            image_paths = []
            for ext in image_extensions:
                image_paths.extend(list(test_dir.rglob(ext)))
            image_paths = [str(p) for p in image_paths[:num_samples]]
        
        if not image_paths:
            print("Warning: No images found for visualization")
            return None
        
        # Run inference and plot
        fig, axes = plt.subplots(1, len(image_paths), figsize=(20, 5))
        if len(image_paths) == 1:
            axes = [axes]
        
        for idx, img_path in enumerate(image_paths):
            # Run inference
            results = model.model(img_path, verbose=False)
            
            # Get the first result
            result = results[0]
            
            # Read image
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Plot on image
            ax = axes[idx]
            ax.imshow(img)
            
            # Draw bounding boxes
            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf = box.conf[0].cpu().numpy()
                    cls = int(box.cls[0].cpu().numpy())
                    
                    # Draw rectangle
                    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                        fill=False, color='red', linewidth=2)
                    ax.add_patch(rect)
                    
                    # Add label
                    class_name = result.names[cls] if hasattr(result, 'names') else f"Class {cls}"
                    label = f"{class_name}: {conf:.2f}"
                    ax.text(x1, y1-5, label, color='red', fontsize=10,
                           bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
            
            ax.set_title(f"Sample {idx+1}", fontsize=12)
            ax.axis('off')
        
        plt.tight_layout()
        return fig
    
    def plot_training_curves(self, training_results):
        """
        Plot training curves from Ultralytics results.
        
        Args:
            training_results: Ultralytics training results object
        
        Returns:
            matplotlib figure
        """
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Try to extract metrics from results
        try:
            # Ultralytics results have results_file attribute
            if hasattr(training_results, 'results_file'):
                results_csv = training_results.results_file
                if Path(results_csv).exists():
                    df = pd.read_csv(results_csv)
                    
                    # Plot train/val loss
                    if 'train/box_loss' in df.columns:
                        axes[0, 0].plot(df['train/box_loss'], label='Train Box Loss', linewidth=2)
                        axes[0, 0].plot(df['val/box_loss'], label='Val Box Loss', linewidth=2)
                        axes[0, 0].set_xlabel('Epoch', fontsize=12)
                        axes[0, 0].set_ylabel('Box Loss', fontsize=12)
                        axes[0, 0].set_title('Box Loss', fontsize=14)
                        axes[0, 0].legend(fontsize=11)
                        axes[0, 0].grid(True, alpha=0.3)
                    
                    # Plot mAP
                    if 'metrics/mAP50(B)' in df.columns:
                        axes[0, 1].plot(df['metrics/mAP50(B)'], label='mAP@0.5', linewidth=2)
                        axes[0, 1].plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', linewidth=2)
                        axes[0, 1].set_xlabel('Epoch', fontsize=12)
                        axes[0, 1].set_ylabel('mAP', fontsize=12)
                        axes[0, 1].set_title('mAP Metrics', fontsize=14)
                        axes[0, 1].legend(fontsize=11)
                        axes[0, 1].grid(True, alpha=0.3)
                    
                    # Plot precision and recall
                    if 'metrics/precision(B)' in df.columns:
                        axes[1, 0].plot(df['metrics/precision(B)'], label='Precision', linewidth=2)
                        axes[1, 0].plot(df['metrics/recall(B)'], label='Recall', linewidth=2)
                        axes[1, 0].set_xlabel('Epoch', fontsize=12)
                        axes[1, 0].set_ylabel('Score', fontsize=12)
                        axes[1, 0].set_title('Precision & Recall', fontsize=14)
                        axes[1, 0].legend(fontsize=11)
                        axes[1, 0].grid(True, alpha=0.3)
                    
                    # Plot F1
                    if 'metrics/F1(B)' in df.columns:
                        axes[1, 1].plot(df['metrics/F1(B)'], label='F1-Score', linewidth=2, color='purple')
                        axes[1, 1].set_xlabel('Epoch', fontsize=12)
                        axes[1, 1].set_ylabel('F1-Score', fontsize=12)
                        axes[1, 1].set_title('F1-Score', fontsize=14)
                        axes[1, 1].legend(fontsize=11)
                        axes[1, 1].grid(True, alpha=0.3)
                    
                    plt.tight_layout()
                    return fig
        except Exception as e:
            print(f"Warning: Could not plot training curves: {e}")
        
        # Fallback: simple plot
        axes[0, 0].text(0.5, 0.5, 'Training curves not available', 
                       ha='center', va='center', fontsize=14)
        for ax in axes.flatten():
            ax.axis('off')
        plt.tight_layout()
        return fig
    
    def plot_pr_curves(self, training_results):
        """
        Plot Precision-Recall curves for each class.
        
        Args:
            training_results: Ultralytics training results object
        
        Returns:
            matplotlib figure
        """
        fig, ax = plt.subplots(figsize=(10, 8))
        
        try:
            # Try to extract PR curves from results
            if hasattr(training_results, 'results_dir'):
                pr_curve_path = Path(training_results.results_dir) / 'PR_curve.png'
                if pr_curve_path.exists():
                    pr_img = plt.imread(pr_curve_path)
                    ax.imshow(pr_img)
                    ax.axis('off')
                    plt.tight_layout()
                    return fig
        except Exception as e:
            print(f"Warning: Could not plot PR curves: {e}")
        
        # Fallback: simple message
        ax.text(0.5, 0.5, 'PR curves not available', 
               ha='center', va='center', fontsize=14)
        ax.axis('off')
        plt.tight_layout()
        return fig

print("✓ DetectionEvaluator defined")
print("\n✓ All YOLOv8 modules loaded successfully!")